# TM1py: Connect, List, Hello World

This module is the second chunk of the tm1py course. Readers are assumed
to have absorbed the three ideas from the previous chunk: tm1py is a
thin wrapper over the TM1 REST API, the library has Objects and
Services, and Python objects are snapshots of server state. With that
mental model in place, this chunk introduces actual code.

The goal is deliberately narrow. By the end, the reader can connect to
a TM1 server and answer the question "what's there?", listing cubes,
dimensions, and processes by name. No object construction, no writes,
no MDX, no DataFrames, no authentication mode beyond basic username and
password. Just open a session and look. The payoff is that within a
ten line script, any TM1 server is browsable from Python.

The chunk covers three pieces of mechanism. First, the `with` block,
Python's standard way to bound a resource's lifetime so that cleanup
always runs. Second, the `TM1Service` facade, which is the connection
plus the namespace of services that hang off it. Third, the universal
`get_all_names()` convention, which works the same way on every Service
in the library. Once those three ideas are routine, the rest of tm1py
is variations on the theme.

The topics below are arranged linearly for review. A single connect-
and-list script grows through the topics, so each section adds one new
element.

---

## Topic list

1. The connection scenario
2. Installing TM1py
3. The TM1Service constructor
4. The with block and why it matters
5. The TM1Service facade
6. The get_all_names convention
7. Hello, world: connect and list
8. Real-world design principles
9. Common mistakes

---

## 1. The connection scenario

Imagine being handed the address of an unfamiliar TM1 server and a set
of credentials, and being asked: "what is on it?" In Architect or in
PAW, the answer is a tree view: cubes, dimensions, processes, chores,
each clickable to drill in. In tm1py, the answer is three method calls
and a few lines of Python.

The exercise is not just procedural. Listing the top-level inventory of
a TM1 server is the simplest possible read: no slicing, no MDX, no
parameters. And yet it exercises every part of how tm1py is shaped.
Opening a connection is the only way to start. Picking a service from
the namespace is the way every later operation begins. Calling a verb
is what every action looks like. Once those three steps are in muscle
memory, the rest of the library is recognizably the same shape applied
to richer operations.

For the running examples through this chunk and the chunks that follow,
imagine a typical small planning model. The cubes include `Sales Plan`,
`General Ledger`, and `HR Plan`. The dimensions include `Period`,
`Region`, `Product`, `Account`, `Version`, and `Measure`. The processes
include things like `Load_Sales` and `Refresh_GL`. The exact contents do
not matter; what matters is that a connect-and-list script lets the
reader see whatever is actually on their own server, with no assumptions
baked in.

Authentication is restricted in this chunk to the basic case: address,
port, user, password, SSL flag. The other modes (CAM, API key,
integrated Windows) are operational topics that are easier to absorb
once the rest of the library is in hand, and they belong to a later
chunk.

## 2. Installing TM1py

tm1py is a pure Python package, distributed on PyPI. Installation is
one line at the shell.

In [ ]:
%%bash
pip install TM1py

There is no compiled component, no native dependency, no driver to
configure. Under the hood the library uses `requests` for HTTP and
parses JSON responses; both come along automatically as dependencies.
The library works against any TM1 server that exposes the REST API,
which means TM1 11 and later, including all current Planning Analytics
releases on premises and on cloud.

Three details about naming are worth fixing in mind early. The
distribution name on PyPI is `TM1py`, with an uppercase T, M, and
lowercase py. The import path is also `TM1py`. The main class on that
import path is `TM1Service`.

In [ ]:
from TM1py import TM1Service

A common opening mistake is to type `import tm1py` (lowercase) and get
a `ModuleNotFoundError`. The casing matches the package name on PyPI;
the file system on case insensitive operating systems often disguises
the issue until the script is moved to Linux.

For production scripts, pin a specific version in `requirements.txt`,
for example `TM1py==2.1.0`. The library is actively maintained and new
versions arrive every few months, occasionally with breaking changes;
pinning protects scripts from version drift. For exploratory work in a
notebook or REPL, `pip install --upgrade TM1py` is fine.

## 3. The TM1Service constructor

The single class that opens a connection is `TM1Service`. Its
constructor takes the connection parameters and, on return, holds an
authenticated session against the server.

In [ ]:
from TM1py import TM1Service

tm1 = TM1Service(
    address="tm1.example.com",
    port=8010,
    user="admin",
    password="apple",
    ssl=True,
)

Each parameter:

- `address` is the host name or IP of the TM1 REST endpoint.
- `port` is the HTTPS port for the TM1 REST API. Common values are 8010
  for an on-premises Planning Analytics installation and a server-
  specific port on Planning Analytics Cloud (the cloud admin console
  shows the URL).
- `user` and `password` are the TM1 credentials.
- `ssl=True` instructs the client to use HTTPS. This should be the
  default for any server beyond a local sandbox. The matching server
  side setting, `UseSSL=T` in `tm1s.cfg`, is on by default for new
  installations.

A successful constructor call returns a live `TM1Service` instance.
"Live" here means: the session has been opened on the server, an
HTTPS connection has been authenticated, and the per-concept services
under the facade (introduced in topic 5) are usable immediately. From
this point until cleanup, every method call on the service goes over
the wire.

The constructor is the only place in tm1py where credentials are
supplied. Once a `TM1Service` exists, code that wants to do something
on the server takes the existing instance rather than re-authenticating.
This mirrors the underlying REST API, which authenticates once per
session and then identifies subsequent calls by a session cookie.

The constructor has no return-without-side-effect form. Building a
`TM1Service` opens a session, full stop. The matching cleanup is
`logout()`, which the next topic shows the right way to call.

## 4. The with block and why it matters

A `TM1Service` holds an open session on the server. Sessions are
finite. Every TM1 server maintains a session table with a maximum
size; once full, the server starts rejecting new connections. Sessions
not explicitly closed by the client time out eventually, but the
default timeout is measured in tens of minutes, which is long enough
for a busy script to accumulate dozens of leaked sessions before any
of them expire.

On Planning Analytics Cloud, this matters more than on a local
on-premises server. Cloud instances are sized to a session limit that
correlates with licensing and is often tighter than local installations.
A development script that crashes ten times in an afternoon may leak
ten sessions into a pool of, say, twenty-five, leaving little room for
other users until the leaked sessions time out. Treating session
cleanup as optional is a cheap habit on a generous on-premises server
and an expensive habit on the cloud.

The disciplined way to guarantee cleanup is the `with` statement.
Python's `with` block bounds a section of code with two operations: a
setup at the start and a cleanup at the end. The cleanup runs when the
block exits, whether normally, by an early return, or by an exception
propagating up through the block.

In [ ]:
with TM1Service(
    address="tm1.example.com",
    port=8010,
    user="admin",
    password="apple",
    ssl=True,
) as tm1:
    print(tm1.server.get_server_name())
# session is closed here, even if the body raised an exception

The setup is the `TM1Service(...)` call, which opens the session. The
cleanup is the implicit `tm1.logout()` that runs when the block ends.
The `as tm1` binds the live service to a name usable inside the block.

The same effect can be expressed with `try` and `finally`, but the
explicit form is longer, easier to forget, and easier to get wrong.

In [ ]:
tm1 = TM1Service(...)
try:
    print(tm1.server.get_server_name())
finally:
    tm1.logout()

The `with` form does the same thing with one keyword. It is also the
Pythonic convention for any resource that needs guaranteed cleanup,
files, database connections, locks, threads, and any tm1py reader will
recognize the pattern from elsewhere in the language. `TM1Service`
supports the protocol that makes `with` work; no extra setup is
required.

The rule that follows is short. For any script intended to run
unattended, the `with` block is mandatory. For interactive work in a
notebook, holding a long-lived `TM1Service` and calling
`tm1.logout()` manually at the end is acceptable, with the understanding
that an unhandled exception in the notebook leaks the session unless
the kernel is restarted.

## 5. The TM1Service facade

The `TM1Service` instance is more than a connection. It is also a
facade, exposing the TM1 model as a fixed tree of attributes. Each
top-level attribute is itself a Service object, scoped to one TM1
concept, and is the entry point for every operation against that
concept.

In [ ]:
with TM1Service(...) as tm1:
    tm1.cubes        # CubeService:        cube CRUD and metadata
    tm1.dimensions   # DimensionService:   dimension CRUD and metadata
    tm1.hierarchies  # HierarchyService:   hierarchies within dimensions
    tm1.elements     # ElementService:     elements within hierarchies
    tm1.subsets      # SubsetService:      named element subsets
    tm1.views        # ViewService:        named MDX or native views
    tm1.cells        # CellService:        cell reads and writes
    tm1.processes    # ProcessService:     TI processes
    tm1.chores       # ChoreService:       scheduled chains of processes
    tm1.security     # SecurityService:    users, groups, permissions
    tm1.applications # ApplicationService: the Apps tree visible in PAW
    tm1.threads      # ThreadService:      live server threads (TM1 Top)
    tm1.sandboxes    # SandboxService:     personal write-back sandboxes
    tm1.server       # ServerService:      server name, config, license

The shape of this namespace is fixed and intentional. Each attribute is
a thin client over a different group of REST endpoints, and the split
mirrors the shape of the TM1 API. There is no surprise here: the
admin's mental map of TM1, with cubes here, dimensions there, processes
in another corner, projects almost one to one onto attributes of
`TM1Service`.

Once the facade is internalized, navigating the library becomes a
two step question. First: which TM1 concept does this operation
concern? Second: what verb on that concept's Service performs it? An
operation on a cube starts with `tm1.cubes`. An operation on a process
starts with `tm1.processes`. Subsets live under `tm1.subsets`, even
when the surrounding workflow is "build a view that uses a subset to
read cells from a cube," because each piece concerns its own concept
and its own service.

The facade does not invent its own model on top of TM1. It does not
have, for example, a unified `tm1.everything` or a query language that
spans concepts. The available operations are what the REST API
exposes, organized by which TM1 concept they act on, and nothing more.
That predictability is part of what makes the library easy to learn
once the shape is known.

## 6. The get_all_names convention

Across every Service in the facade, the same method appears with the
same signature: `get_all_names()`. It takes no required arguments and
returns a Python list of strings, the names of every object of that
kind on the server. It is the simplest possible read.

In [ ]:
with TM1Service(...) as tm1:
    tm1.cubes.get_all_names()
    # ['Sales Plan', 'General Ledger', 'HR Plan', 'Workforce Plan']

    tm1.dimensions.get_all_names()
    # ['Period', 'Region', 'Product', 'Account', 'Version', 'Measure', ...]

    tm1.processes.get_all_names()
    # ['Load_Sales', 'Refresh_GL', 'Archive_Quarter', ...]

    tm1.chores.get_all_names()
    # ['Nightly_Refresh', 'End_Of_Month']

The method exists on every Service that fronts a named TM1 concept:
cubes, dimensions, hierarchies, subsets, views, processes, chores, and
the rest. The reader who has internalized the convention does not need
to consult documentation to ask "what subsets exist on this server?"
The answer is `tm1.subsets.get_all_names()`, by analogy. The library is
predictable on purpose.

A close cousin, `get_all()`, returns full Object instances rather than
names. It is correspondingly slower and produces a much larger payload,
since each Object includes attributes, dimensions, rules text, and the
rest. When the question is just "what exists," `get_all_names()` is the
right answer. When the question is "what exists and what does each one
look like," `get_all()` is the right answer, paid for by a heavier
request.

The pattern of naming verbs to imply their cost is a recurring theme in
tm1py and inherits directly from the REST API design. `get_all_names()`
is a metadata read that returns names only. `get_all()` returns full
objects. `get(name)` returns one object by name. `exists(name)` returns
a boolean. Each is the smallest read that answers its specific
question.

For the connect and list script that this chunk is building toward, the
right verb is `get_all_names()`, three times, against three different
services. The combined output is the answer to "what is on this
server?"

## 7. Hello, world: connect and list

Putting every previous topic together, the entire chunk fits in a
single runnable script.

In [ ]:
from TM1py import TM1Service

with TM1Service(
    address="tm1.example.com",
    port=8010,
    user="admin",
    password="apple",
    ssl=True,
) as tm1:
    print("Cubes:")
    for name in tm1.cubes.get_all_names():
        print(f"  {name}")

    print("\nDimensions:")
    for name in tm1.dimensions.get_all_names():
        print(f"  {name}")

    print("\nProcesses:")
    for name in tm1.processes.get_all_names():
        print(f"  {name}")

Sample output against the running model:

```
Cubes:
  Sales Plan
  General Ledger
  HR Plan
  Workforce Plan

Dimensions:
  Period
  Region
  Product
  Account
  Version
  Measure
  Currency

Processes:
  Load_Sales
  Refresh_GL
  Archive_Quarter
  Nightly_Refresh
```

Every piece of the script has appeared in an earlier topic. The
`from TM1py import TM1Service` line is the import (topic 2). The
`TM1Service(...)` constructor opens the session (topic 3). The `with`
block guarantees `logout()` (topic 4). The dotted access into
`tm1.cubes`, `tm1.dimensions`, and `tm1.processes` is the facade
(topic 5). The `get_all_names()` calls are the universal verb
(topic 6). The `for` loops are ordinary Python.

This is a complete, working tm1py program. It connects, queries three
inventories, prints them, and disconnects cleanly. Every later chunk
in this course extends this skeleton, replacing the trivial body with
richer reads, transformations, and writes, but the outline does not
change.

The reader's first task is to run this script against a real TM1
server. The combination of the `with` block bounding the session, the
facade selecting the service, and `get_all_names()` doing the read is
the smallest tm1py program that does anything useful. Producing real
output from a real server is the moment the library stops being
abstract.

## 8. Real-world design principles

Once the connect-and-list pattern is in hand, a small number of
guidelines apply to almost every tm1py script that grows out of it.

**Always use `with TM1Service(...) as tm1`.** Topic 4 covered the
mechanics. The principle is that any code path that does not reach
`tm1.logout()` leaks a session, and on Planning Analytics Cloud that
matters quickly. The `with` block is the only form that survives
exceptions cleanly, and at one keyword it costs nothing.

**Build the connection once per script and reuse it.** A new
`TM1Service` is a new login round trip and a new server-side session
slot. Inside a single script, open one `with` block at the top, do all
the work inside it, and let the block close when the script ends. The
cost of the alternative, building a new service per call, is paid in
network round trips and in session slots; it dominates the runtime of
otherwise simple scripts.

**Prefer `get_all_names()` over `get_all()` when the question is "what
exists."** Names are smaller, faster to fetch, and almost always
sufficient. Reach for `get_all()` only when the body of every Object is
needed, which is rare for inventory questions and common for, say,
dumping every dimension's full structure to disk.

**Cache results of cheap reads when used repeatedly.** Per the snapshot
principle from chunk 1, even `get_all_names()` is a network round trip,
not a free local lookup. If the same list is consulted three times
inside a function, fetch it once at the top and pass it around. The
discipline is the same one TM1 administrators apply to TI: hit the
server fewer times when the answer would not change.

**Keep credentials out of source code.** Read `address`, `user`, and
`password` from environment variables or from a secrets store. The
small extra effort at the start prevents a credential leak when a
script is committed to git, shared in a notebook, or pasted into a
chat. This is not a tm1py concern specifically; it is general practice
for any code that connects to a server.

## 9. Common mistakes

A short collection of errors that are easy to make on the way to a
working connect-and-list script, and worth recognizing early.

**Forgetting the `with` block.** A bare `TM1Service(...)` followed by
ad hoc work leaks the session on any exception. The remedy is the
`with` block.

In [ ]:
# Wrong: leaks the session if anything between raises
tm1 = TM1Service(address=..., user=..., password=..., port=..., ssl=True)
print(tm1.cubes.get_all_names())
tm1.logout()

# Correct
with TM1Service(address=..., user=..., password=..., port=..., ssl=True) as tm1:
    print(tm1.cubes.get_all_names())

**Calling `get_all()` when only names are needed.** The full read pulls
every Object's full body, including attributes, dimensions, and rules.
For an inventory question this is wasted bandwidth.

In [ ]:
# Wrong: pulls the full Cube object for every cube
for cube in tm1.cubes.get_all():
    print(cube.name)

# Correct: pulls only the names
for name in tm1.cubes.get_all_names():
    print(name)

**Hardcoding credentials into source.** A password literal in a script
is a credential about to be committed to git. Read from the environment
instead.

In [ ]:
# Wrong: password ends up in the repository
tm1 = TM1Service(
    address="tm1.example.com", port=8010,
    user="admin", password="apple", ssl=True,
)

# Correct
import os

tm1 = TM1Service(
    address=os.environ["TM1_ADDRESS"],
    port=int(os.environ["TM1_PORT"]),
    user=os.environ["TM1_USER"],
    password=os.environ["TM1_PASSWORD"],
    ssl=True,
)

**Reusing a service object after the `with` block exits.** Once the
block ends, the session is closed. Calls on the variable from outside
the block fail or behave unpredictably depending on the tm1py version.

In [ ]:
# Wrong: tm1 is logged out by the time print runs
with TM1Service(...) as tm1:
    cubes = tm1.cubes.get_all_names()
print(tm1.cubes.get_all_names())   # session is gone

# Correct: do the work inside the block
with TM1Service(...) as tm1:
    cubes = tm1.cubes.get_all_names()
    print(cubes)

**Opening a new `TM1Service` per call inside a loop.** Each iteration
pays for a fresh login and a fresh session slot, neither of which is
free. Open one connection, loop inside the block.

In [ ]:
# Wrong: one login per iteration
for cube_name in cube_names:
    with TM1Service(...) as tm1:
        print(tm1.cubes.get(cube_name).dimensions)

# Correct: one login covers the whole loop
with TM1Service(...) as tm1:
    for cube_name in cube_names:
        print(tm1.cubes.get(cube_name).dimensions)

**Importing the package as `tm1py` (lowercase).** The distribution and
import name is `TM1py`, with capital T, M, and the digit 1, followed by
lowercase py. The wrong casing produces a `ModuleNotFoundError`.

In [ ]:
# Wrong
import tm1py                   # ModuleNotFoundError

# Correct
from TM1py import TM1Service